# CS231n 2026 A1 Q1 · k近傍法を実装する

`CS231N26-A1-Q01` · 学習用Colabノート（未解答）

[Stanford公式の課題ページ](https://cs231n.github.io/assignments2026/assignment1/) · [公式スターターZIP](https://cs231n.github.io/assignments/2026/assignment1.zip)

**目標**：訓練例を保存するkNN分類器を作り、画像間距離を2重ループ・1重ループ・行列演算で計算する。多数決で予測し、訓練データだけを使う交差検証で `k` を選ぶ。

対応する実装箇所：`CS231N26-A1-Q01-TODO01`（距離と予測）、`CS231N26-A1-Q01-TODO02`（交差検証）。

最初は軽量な手書き数字データで各実装を確かめる。最後の節で、公式課題が使用するCIFAR-10へ進む。数値条件と提出条件は公式課題が優先です。ここに模範解答は入れていません。


## 0. 準備とデータ分割

手書き数字の各画像を1行の特徴ベクトルにして、訓練用とテスト用に分けます。テスト用の正解ラベルを `k` の選択に使わないでください。まず下のセルを実行し、形状を確認します。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data.astype(np.float64), digits.target,
    test_size=0.2, stratify=digits.target, random_state=42,
)
print('train:', X_train.shape, 'test:', X_test.shape)
assert X_train.shape[1] == X_test.shape[1] == 64


## 1. 距離計算と予測を実装

入力 `X` は `(テスト件数, 特徴数)`、保存した `self.X_train` は `(訓練件数, 特徴数)` です。各距離関数は `(テスト件数, 訓練件数)` の行列を返します。

- 2重ループ：各組の差を取り、L2距離を計算する。
- 1重ループ：テスト例ごとに、訓練例すべてとの距離を一括計算する。
- ループなし：内積と二乗ノルムを用いて距離行列を作る。浮動小数誤差で負になった二乗距離は0に丸める。
- 予測：近い `k` 件のラベルで多数決する。同票時は小さいラベルを採用する。

公式の距離計算課題には追加の制約があります。最終的な提出実装では公式スターターの指示を確認してください。


In [ ]:
class KNN:
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        if X.ndim != 2 or len(X) != len(y):
            raise ValueError('Xとyの件数を確認してください')
        self.X_train = X
        self.y_train = y
        return self

    def distances_two_loops(self, X):
        # TODO 1: (len(X), len(self.X_train)) のL2距離行列を返す
        raise NotImplementedError('2重ループの距離計算を実装')

    def distances_one_loop(self, X):
        # TODO 2: テスト例だけをループし、訓練側はNumPyで一括計算
        raise NotImplementedError('1重ループの距離計算を実装')

    def distances_no_loops(self, X):
        # TODO 3: 行列積で全組の二乗距離を求め、平方根を取る
        raise NotImplementedError('ループなしの距離計算を実装')

    def predict_labels(self, distances, k=1):
        # TODO 4: 各行の最近傍k件を選び、多数決でラベルを返す
        raise NotImplementedError('多数決を実装')

    def predict(self, X, k=1):
        distances = self.distances_no_loops(np.asarray(X, dtype=np.float64))
        return self.predict_labels(distances, k=k)


## 2. 小さな例で検算

実装が終わったら実行してください。距離行列の形、既知の距離、3方式の一致、多数決を確認します。失敗した行が修正箇所の手掛かりです。


In [ ]:
tiny = KNN().fit(
    np.array([[0., 0.], [3., 4.], [0., 5.]]),
    np.array([0, 1, 1]),
)
queries = np.array([[0., 0.], [3., 0.]])
expected = np.array([[0., 5., 5.], [3., 4., np.sqrt(34.)]])
d2 = tiny.distances_two_loops(queries)
d1 = tiny.distances_one_loop(queries)
d0 = tiny.distances_no_loops(queries)
assert d2.shape == (2, 3)
np.testing.assert_allclose(d2, expected, atol=1e-10)
np.testing.assert_allclose(d1, expected, atol=1e-10)
np.testing.assert_allclose(d0, expected, atol=1e-10)
np.testing.assert_array_equal(tiny.predict_labels(d0, k=1), [0, 0])
print('距離3方式とk=1の予測が一致しました')


In [ ]:
# 距離行列を画像として確認する。明るい行・列が示すものを考える。
plt.figure(figsize=(8, 4))
plt.imshow(d0, interpolation='nearest', aspect='auto')
plt.xlabel('train index')
plt.ylabel('test index')
plt.colorbar(label='L2 distance')
plt.show()


## 3. 手書き数字の分類

`k=1` と `k=5` のテスト精度を計算し、数件の誤分類を画像で見てください。距離行列の行や列が明るくなる理由も説明してみましょう。


In [ ]:
model = KNN().fit(X_train, y_train)
for k in (1, 5):
    predicted = model.predict(X_test, k=k)
    accuracy = np.mean(predicted == y_test)
    print(f'k={k}: test accuracy={accuracy:.3f}')

# TODO: 誤分類例を数件選び、8×8の画像と予測/正解ラベルを表示する


## 4. 交差検証でkを選ぶ

訓練データを5分割し、各 `k` の検証精度を5つ記録します。最良の `k` を決めてから、保持していたテストデータで**一度だけ**評価します。毎回の学習に検証foldを混ぜないことがポイントです。


In [ ]:
k_choices = [1, 3, 5, 7, 9]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {k: [] for k in k_choices}

for fit_idx, valid_idx in folds.split(X_train, y_train):
    fold_model = KNN().fit(X_train[fit_idx], y_train[fit_idx])
    distances = fold_model.distances_no_loops(X_train[valid_idx])
    for k in k_choices:
        # TODO: predict_labelsで予測し、検証foldの精度をcv_scores[k]へ追加
        pass

# TODO: k別の平均・標準偏差を描いてbest_kを選ぶ
# TODO: X_train全体で再学習し、X_testで最終精度を測る


## 5. 公式CIFAR-10課題へ進む（データ量に注意）

公式のQ1はCIFAR-10の最初の訓練5,000枚・テスト500枚を用います。公式[Assignment 1](https://cs231n.github.io/assignments2026/assignment1/)の `knn.ipynb` と `cs231n/classifiers/k_nearest_neighbor.py` には、提出用の指示・TODOがあります。このノートで検算したロジックを公式スターターへ移し、指定された制約と設問に従って完成させてください。

下のセルはデータ取得だけを行います。CIFAR-10のダウンロードは約160MBです。2重ループを全5,000×500組で動かすと時間がかかるため、最初は小さい部分集合で検算します。


In [ ]:
# 必要なときだけ実行: CIFAR-10 Python版のデータ取得
import pathlib, pickle, tarfile, urllib.request

data_dir = pathlib.Path('/content/cifar-10-batches-py')
archive = pathlib.Path('/content/cifar-10-python.tar.gz')
if not data_dir.exists():
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', archive
    )
    with tarfile.open(archive, 'r:gz') as tar:
        # アーカイブ内の想定ディレクトリだけを展開する
        members = [m for m in tar.getmembers()
                   if m.name == 'cifar-10-batches-py'
                   or m.name.startswith('cifar-10-batches-py/')]
        tar.extractall('/content', members=members, filter='data')

def read_cifar_batch(path):
    with open(path, 'rb') as stream:
        batch = pickle.load(stream, encoding='bytes')
    return batch[b'data'].astype(np.float64), np.asarray(batch[b'labels'])

X_cifar_train, y_cifar_train = read_cifar_batch(data_dir / 'data_batch_1')
X_cifar_test, y_cifar_test = read_cifar_batch(data_dir / 'test_batch')
X_cifar_train, y_cifar_train = X_cifar_train[:5000], y_cifar_train[:5000]
X_cifar_test, y_cifar_test = X_cifar_test[:500], y_cifar_test[:500]
print(X_cifar_train.shape, X_cifar_test.shape)
assert X_cifar_train.shape == (5000, 3072)
assert X_cifar_test.shape == (500, 3072)


## 学習メモ

- 3種類の距離実装の計算時間とメモリ使用量はどう違ったか。
- `k` を大きくすると訓練精度と検証精度はどう変化したか。
- 交差検証と最終テストを分けた理由は何か。
- 公式の画像データで誤分類が起こる例には、どんな共通点があるか。

**参考**：[Stanford公式Assignment 1](https://cs231n.github.io/assignments2026/assignment1/)。このノートは独自の学習補助で、公式提出物そのものではありません。
